# TOB予測モデル SHAP分析

論文「機械学習による他社株TOBの予測可能性」の再現 — 特徴量寄与度分析。
`train_rf.py` のキャッシュデータを再利用してモデルを再構築し、SHAP値を計算・可視化する。

In [ ]:
import sys
import subprocess
from pathlib import Path


def setup_runtime():
    runtime = 'local'
    auth_mod = None
    userdata_mod = None
    try:
        from google.colab import auth as colab_auth, userdata as colab_userdata
        runtime = 'colab'
        auth_mod = colab_auth
        userdata_mod = colab_userdata
    except ImportError:
        pass

    if runtime == 'colab':
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'shap', 'imbalanced-learn', 'japanize-matplotlib', 'structlog', 'optuna'],
            capture_output=True,
            check=False,
        )
        from google.colab import drive
        drive.mount('/content/drive')
        project_root = Path('/content/drive/MyDrive/claude/investment-agent')
        cache_dir = Path('/content/tob_prediction')
    else:
        project_root = Path('C:/gdrive/claude/investment-agent')
        cache_dir = Path('C:/tmp/tob_prediction')

    cache_dir.mkdir(parents=True, exist_ok=True)
    return runtime, auth_mod, userdata_mod, project_root, cache_dir


RUNTIME, auth, userdata, PROJECT_ROOT, CACHE_DIR = setup_runtime()

# train_rf.py を import path に追加
sys.path.insert(0, str(PROJECT_ROOT / 'scripts' / 'tob_prediction'))

import numpy as np
import pandas as pd
import shap
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTENC

# --- 日本語フォント ---
if RUNTIME == 'colab':
    import japanize_matplotlib
    matplotlib.rcParams['axes.unicode_minus'] = False
else:
    plt.rcParams["font.family"] = "MS Gothic"

# train_rf.py の関数・定数を再利用（CACHE_DIR を上書き）
import train_rf
train_rf.CACHE_DIR = CACHE_DIR
from train_rf import (
    build_feature_matrix,
    CONTINUOUS_FEATURES,
    BINARY_FEATURES,
    _cached, _bq_client,
    _load_labels, _load_financials, _load_shareholders,
    _load_price_features,
)

plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 100
print(f"RUNTIME={RUNTIME}, CACHE_DIR={CACHE_DIR}")

## 1. データロード（キャッシュ再利用）

In [ ]:
# --- BQ client (runtime-dependent) ---
if 'RUNTIME' not in globals():
    RUNTIME, auth, userdata, PROJECT_ROOT, CACHE_DIR = setup_runtime()

if RUNTIME == 'colab':
    if auth is None:
        raise RuntimeError("google.colab.auth の初期化に失敗しました。先頭セルを再実行してください。")
    auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project='gmailpj-357912')
else:
    client = _bq_client()

# --- Load with cache (BQ fallback if cache miss) ---
labels = _cached("labels", _load_labels, client, refresh=False)
financials = _cached("financials", _load_financials, client, refresh=False)
shareholders = _cached("shareholders", _load_shareholders, client, refresh=False)
prices = _cached("prices", _load_price_features, client, refresh=False)

print(f"Labels: {len(labels)}, Financials: {len(financials)}, Shareholders: {len(shareholders)}")
print(f"Prices: {len(prices)}")

features = build_feature_matrix(financials, shareholders, prices, labels)
print(f"\nFeature matrix: {features.shape}")
print(f"Positives: {features['label'].sum():.0f} / {len(features)}")

## 2. モデル構築（2025年評価、訓練データ最大）

In [ ]:
EVAL_YEAR = 2025
TRAIN_WINDOW = 5

feature_cols = CONTINUOUS_FEATURES + BINARY_FEATURES
n_cont = len(CONTINUOUS_FEATURES)
cat_indices = list(range(n_cont, n_cont + len(BINARY_FEATURES)))

# 日本語の特徴量名マッピング
FEATURE_NAMES_JA = {
    "equity_ratio": "自己資本比率",
    "pbr": "PBR",
    "roe": "ROE(実績)",
    "payout_ratio": "配当性向",
    "ln_market_cap": "log(時価総額)",
    "cash_rich_ratio": "キャッシュリッチ比率",
    "forecast_div_yield": "予想配当利回り",
    "forecast_profit_growth": "予想利益成長率",
    "cfo_to_mcap": "CF/時価総額",
    "operating_margin": "営業利益率",
    "ret_60d": "60日リターン",
    "ret_240d": "240日リターン",
    "vol_240d": "240日ボラティリティ",
    "turnover_ratio": "出来高回転率",
    "top_shareholder_ratio": "筆頭株主比率",
    "individual_ratio": "個人持株比率",
    "foreign_ratio": "外国人持株比率",
    "financial_inst_ratio": "金融機関持株比率",
    "other_corp_ratio": "法人持株比率",
    "top10_concentration": "上位10株主集中度",
    "has_activist": "アクティビスト有無",
    "top_shareholder_is_public": "筆頭株主上場",
}
feature_names_display = [FEATURE_NAMES_JA.get(c, c) for c in feature_cols]

# Split
train_years = list(range(EVAL_YEAR - TRAIN_WINDOW, EVAL_YEAR))
df_train = features[features["year"].isin(train_years)].dropna(subset=feature_cols)
df_test = features[features["year"] == EVAL_YEAR].dropna(subset=feature_cols)

X_train = df_train[feature_cols].values.astype(np.float64)
y_train = df_train["label"].values.astype(int)
X_test = df_test[feature_cols].values.astype(np.float64)
y_test = df_test["label"].values.astype(int)

print(f"Train: {len(X_train)} (pos={y_train.sum()})")
print(f"Test:  {len(X_test)} (pos={y_test.sum()})")

# Standardize continuous features
scaler = StandardScaler()
X_train[:, :n_cont] = scaler.fit_transform(X_train[:, :n_cont])
X_test[:, :n_cont] = scaler.transform(X_test[:, :n_cont])

# Resample (6-step preprocessing)
rus = RandomUnderSampler(sampling_strategy=0.05, random_state=42)
X_r, y_r = rus.fit_resample(X_train, y_train)
tl = TomekLinks()
X_r, y_r = tl.fit_resample(X_r, y_r)

minority_count = int(y_r.sum())
if minority_count >= 2:
    smote = SMOTENC(
        categorical_features=cat_indices,
        sampling_strategy=0.1,
        random_state=42,
        k_neighbors=min(5, minority_count - 1),
    )
    X_r, y_r = smote.fit_resample(X_r, y_r)
    print(f"After SMOTENC: {len(X_r)} (pos={y_r.sum()})")
else:
    print(f"Skip SMOTENC: minority_count={minority_count}")

# Train RF (2025 best params from walk-forward)
clf = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_leaf=14,
    max_features="sqrt", random_state=42, n_jobs=-1,
)
clf.fit(X_r, y_r)

from sklearn.metrics import roc_auc_score, average_precision_score
y_prob = clf.predict_proba(X_test)[:, 1]
print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")

## 3. SHAP値計算

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# shap の返り値は version / model により list, ndarray(3D), Explanation など差分がある
if isinstance(shap_values, list):
    sv = shap_values[1]
elif hasattr(shap_values, "values"):
    sv = shap_values.values
else:
    sv = shap_values

if getattr(sv, "ndim", None) == 3:
    sv = sv[:, :, 1]

if getattr(sv, "ndim", None) != 2:
    raise ValueError(f"Unexpected SHAP shape: {getattr(sv, 'shape', None)}")

print(f"SHAP values shape: {sv.shape}")
print(f"Test samples: {len(X_test)}, Features: {len(feature_cols)}")

## 4. SHAP Summary Plot（全特徴量の寄与度ビースウォーム）

In [ ]:
shap.summary_plot(sv, X_test, feature_names=feature_names_display, max_display=20, show=False)
plt.title("SHAP Summary — TOB予測 (2025年テスト)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. SHAP Bar Plot（平均絶対SHAP値）

In [ ]:
shap.summary_plot(sv, X_test, feature_names=feature_names_display, plot_type="bar", max_display=20, show=False)
plt.title("特徴量重要度（平均|SHAP|）— TOB予測 (2025年)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Dependence Plots（論文の重要特徴量 Top4）

論文 SHAP 分析の重要変数: 筆頭株主比率 → 筆頭株主上場 → 個人持株比率 → log(時価総額)

In [ ]:
key_features = [
    ("top_shareholder_ratio", "筆頭株主比率"),
    ("top_shareholder_is_public", "筆頭株主上場"),
    ("individual_ratio", "個人持株比率"),
    ("ln_market_cap", "log(時価総額)"),
    ("other_corp_ratio", "法人持株比率"),
    ("has_activist", "アクティビスト有無"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (feat, ja_name) in zip(axes.flat, key_features):
    idx = feature_cols.index(feat)
    plt.sca(ax)
    shap.dependence_plot(
        idx, sv, X_test,
        feature_names=feature_names_display,
        show=False, ax=ax,
    )
    ax.set_title(f"{ja_name}", fontsize=12)

plt.suptitle("SHAP Dependence — 主要特徴量", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- Dependence data export (CSV) ---
dep_rows = []
for feat, ja_name in key_features:
    idx = feature_cols.index(feat)
    for j in range(len(X_test)):
        dep_rows.append({
            "feature": ja_name,
            "feature_value": X_test[j, idx],
            "shap_value": sv[j, idx],
            "label": y_test[j],
        })

dep_df = pd.DataFrame(dep_rows)
if RUNTIME == "colab":
    dep_path = "/content/shap_dependence.csv"
else:
    dep_path = str(CACHE_DIR / "shap_dependence.csv")
dep_df.to_csv(dep_path, index=False, encoding="utf-8")
print(f"Dependence data saved: {dep_path} ({len(dep_df)} rows)")


## 7. HAS_ACTIVIST 固有寄与度分析

論文にない追加因子。アクティビスト保有がTOB予測にどう効くか。

In [ ]:
activist_idx = feature_cols.index("has_activist")
public_idx = feature_cols.index("top_shareholder_is_public")


def safe_mean(arr):
    return float(np.mean(arr)) if len(arr) else float("nan")


# アクティビスト有無別の予測確率
mask_activist = X_test[:, activist_idx] == 1
mask_no_activist = X_test[:, activist_idx] == 0

print("=== HAS_ACTIVIST の効果 ===")
print(f"アクティビストあり: {mask_activist.sum()} 社, TOB実績 {y_test[mask_activist].sum():.0f} 件, "
      f"TOB率 {safe_mean(y_test[mask_activist]):.3%}")
print(f"アクティビストなし: {mask_no_activist.sum()} 社, TOB実績 {y_test[mask_no_activist].sum():.0f} 件, "
      f"TOB率 {safe_mean(y_test[mask_no_activist]):.3%}")
print(f"\n平均予測確率:")
print(f"  アクティビストあり: {safe_mean(y_prob[mask_activist]):.4f}")
print(f"  アクティビストなし: {safe_mean(y_prob[mask_no_activist]):.4f}")
print(f"\n平均SHAP値 (HAS_ACTIVIST):")
print(f"  アクティビストあり: {safe_mean(sv[mask_activist, activist_idx]):.6f}")
print(f"  アクティビストなし: {safe_mean(sv[mask_no_activist, activist_idx]):.6f}")
print(f"\n全特徴量中の重要度ランク: ", end="")
mean_abs_shap = np.abs(sv).mean(axis=0)
ranking = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
rank = list(ranking.index).index("has_activist") + 1
print(f"{rank}位 / {len(feature_cols)}特徴量")

## 8. 論文比較テーブル（SHAP重要度ランキング）

In [ ]:
paper_ranking = {
    "top_shareholder_ratio": 1,
    "top_shareholder_is_public": 2,
    "individual_ratio": 3,
    "ln_market_cap": 4,
    "pbr": 5,
    "ret_240d": 6,
    "payout_ratio": 7,
}

comparison = []
for i, (feat, shap_val) in enumerate(ranking.items()):
    ja = FEATURE_NAMES_JA.get(feat, feat)
    paper_rank = paper_ranking.get(feat, "-")
    comparison.append({
        "実装ランク": i + 1,
        "特徴量": ja,
        "平均|SHAP|": f"{shap_val:.6f}",
        "論文ランク": paper_rank,
    })

comp_df = pd.DataFrame(comparison[:15])
print("=== SHAP重要度: 実装 vs 論文 ===")
print(comp_df.to_string(index=False))

## 9. SHAP Interaction（筆頭株主比率 × 主要特徴量）

筆頭株主比率が他の特徴量とどう相互作用するか。

In [ ]:
# Interaction: top_shareholder_ratio と主要変数
interaction_pairs = [
    ("top_shareholder_ratio", "top_shareholder_is_public"),
    ("top_shareholder_ratio", "pbr"),
    ("top_shareholder_ratio", "ln_market_cap"),
    ("top_shareholder_ratio", "has_activist"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (f1, f2) in zip(axes.flat, interaction_pairs):
    idx1 = feature_cols.index(f1)
    idx2 = feature_cols.index(f2)
    ja1 = FEATURE_NAMES_JA.get(f1, f1)
    ja2 = FEATURE_NAMES_JA.get(f2, f2)
    scatter = ax.scatter(
        X_test[:, idx1], sv[:, idx1],
        c=X_test[:, idx2], cmap="coolwarm", alpha=0.5, s=10,
    )
    ax.set_xlabel(ja1)
    ax.set_ylabel(f"SHAP({ja1})")
    cb = plt.colorbar(scatter, ax=ax)
    cb.set_label(ja2)
    ax.set_title(f"{ja1} × {ja2}")

plt.suptitle("SHAP Interaction — 筆頭株主比率と他変数の交互作用", fontsize=14)
plt.tight_layout()
plt.show()


# --- Interaction data export (CSV) ---
int_rows = []
for f1, f2 in interaction_pairs:
    idx1 = feature_cols.index(f1)
    idx2 = feature_cols.index(f2)
    ja1 = FEATURE_NAMES_JA.get(f1, f1)
    ja2 = FEATURE_NAMES_JA.get(f2, f2)
    for j in range(len(X_test)):
        int_rows.append({
            "pair": f"{ja1} x {ja2}",
            "feature1": ja1,
            "feature1_value": X_test[j, idx1],
            "shap_feature1": sv[j, idx1],
            "feature2": ja2,
            "feature2_value": X_test[j, idx2],
            "label": y_test[j],
        })

int_df = pd.DataFrame(int_rows)
if RUNTIME == "colab":
    int_path = "/content/shap_interaction.csv"
else:
    int_path = str(CACHE_DIR / "shap_interaction.csv")
int_df.to_csv(int_path, index=False, encoding="utf-8")
print(f"Interaction data saved: {int_path} ({len(int_df)} rows)")


## 10. 2025年 Top予測銘柄プロファイリング

予測確率上位10銘柄のSHAP Waterfall — 各銘柄がなぜ高スコアなのかを分解。

In [ ]:
# Top10 予測銘柄の Waterfall
ranking_df = (
    df_test[["TICKER"]].copy()
    .assign(prob=y_prob, label=y_test)
    .sort_values("prob", ascending=False)
    .reset_index(drop=True)
)

print("=== 2025年 予測確率 Top10 ===")
print(ranking_df.head(10).to_string(index=False))

top_indices = ranking_df.head(10).index.tolist()

explanation = shap.Explanation(
    values=sv[top_indices],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    data=X_test[top_indices],
    feature_names=feature_names_display,
)

fig, axes = plt.subplots(2, 5, figsize=(28, 10))
for i, ax in enumerate(axes.flat):
    if i >= len(top_indices):
        ax.axis("off")
        continue
    ticker = ranking_df.iloc[top_indices[i]]["TICKER"]
    prob = ranking_df.iloc[top_indices[i]]["prob"]
    is_tob = "★TOB" if ranking_df.iloc[top_indices[i]]["label"] == 1 else ""
    plt.sca(ax)
    shap.plots.waterfall(explanation[i], max_display=10, show=False)
    ax.set_title(f"{ticker} (p={prob:.3f}) {is_tob}", fontsize=10)

plt.suptitle("SHAP Waterfall — 2025年 予測確率 Top10", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 11. 予測精度サマリー（Top N% ヒット率）

In [ ]:
# Top N% のヒット率
print("=== 予測確率上位 N% のTOBヒット率 ===")
print(f"{'Top%':>6} {'銘柄数':>6} {'TOB件数':>7} {'ヒット率':>8} {'ベース率':>8}")
base_rate = y_test.mean()
for pct in [1, 3, 5, 10, 15, 25]:
    n = max(1, int(len(ranking_df) * pct / 100))
    top = ranking_df.head(n)
    hits = int(top["label"].sum())
    hit_rate = hits / n
    lift = hit_rate / base_rate if base_rate > 0 else 0
    print(f"{pct:>5}% {n:>6} {hits:>7} {hit_rate:>8.1%} {lift:>7.1f}x")

print()
print(f"ベース率 (全体TOB率): {base_rate:.2%}")
print(f"テスト銘柄数: {len(ranking_df)}, TOB件数: {int(y_test.sum())}")